# GPG + Cooldown：Google Colab 實驗筆記本（L4 / A100 40G）

照順序執行。每次開新的 Colab session 都要從 **步驟 1** 重跑到 **步驟 4**（安裝套件只需 1–2 分鐘）。

- **程式碼與實驗輸出放在 Google Drive**（`MyDrive/GPG-Cooldown`）：Colab 斷線後 checkpoint 不會消失，重跑同一個指令會自動從最後一個 checkpoint 續訓（每 50 步存一次，冷卻狀態也會一併還原）。
- **模型權重放在 Colab 本機磁碟**（`/content/models`）：讀取比 Drive 快，每個 session 重新下載一次。
- 4-bit（QLoRA）在 Linux 上可直接用 `bitsandbytes`，加 `--load_in_4bit true` 即可。
- **公平比較**：同一組比較內（GPG 原版 vs Cooldown）的 GPU、4-bit 與 LoRA 設定必須一致；不同 GPU 的耗時不要互相比。

## 步驟 1：確認 GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 步驟 2：掛載 Google Drive 並取得程式碼

若 repo 是 private，請先在左側 🔑「Secrets」新增 `GITHUB_TOKEN`（GitHub 的 Personal Access Token，需 repo 讀取權限），並開啟此筆記本的存取權。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
REPO_DIR = '/content/drive/MyDrive/GPG-Cooldown'
BRANCH = 'main'  # 要測試其他分支時改這裡

if not os.path.isdir(REPO_DIR):
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        url = f'https://{token}@github.com/chris456621/GPG-Cooldown.git'
    except Exception:
        url = 'https://github.com/chris456621/GPG-Cooldown.git'
    subprocess.run(['git', 'clone', '-b', BRANCH, url, REPO_DIR], check=True)
    # 不把 token 留在 Drive 上的 .git/config
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', 'https://github.com/chris456621/GPG-Cooldown.git'], check=True)
else:
    try:
        from google.colab import userdata
        remote = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/chris456621/GPG-Cooldown.git"
    except Exception:
        remote = 'origin'
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', remote, BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'merge', '--ff-only', 'FETCH_HEAD'], check=True)

OPEN_R1 = f'{REPO_DIR}/open-r1'
%cd {OPEN_R1}
!git log --oneline -3

## 步驟 3：安裝套件

沿用 Colab 內建的 PyTorch（不重裝），其餘套件依 `experiment_config.yaml` 鎖定的版本安裝，另外加裝 `bitsandbytes`（4-bit 用）。
Colab 的 Python / PyTorch 版本與 Windows 設定不同，環境檢查會顯示 `MISMATCH` 警告，這是預期的，不影響訓練。

In [ ]:
import yaml
cfg = yaml.safe_load(open('experiment_config.yaml', encoding='utf-8'))
pkgs = ' '.join(f'{k}=={v}' for k, v in cfg['environment']['packages'].items())
!pip install -q {pkgs} bitsandbytes

## 步驟 4：設定環境變數並把模型目錄放到本機磁碟

In [ ]:
import os
os.environ['PYTHONPATH'] = 'src'
os.environ['PYTHONUTF8'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HOME'] = '/content/hf_cache'

# open-r1/models -> /content/models（本機 SSD，避免在 Drive 上讀寫數十 GB 權重）
os.makedirs('/content/models', exist_ok=True)
if os.path.islink('models') or not os.path.exists('models'):
    if os.path.islink('models'):
        os.unlink('models')
    os.symlink('/content/models', 'models')
else:
    print('注意：open-r1/models 是實體資料夾（在 Drive 上），模型會從 Drive 讀取，速度較慢。')

# 需要 Llama 等需授權的模型時再執行：
# from huggingface_hub import login; login()

## 步驟 5：煙霧測試（約 5–10 分鐘）

用 0.5B + 4-bit 只跑 5 步，確認整條流程（下載、4-bit 載入、訓練、評測、寫報告）在 Colab 上都能跑通。成功後才開始正式實驗。

In [ ]:
!python scripts/compare_gpg_vs_cooldown.py \
  --model_name_or_path models/Qwen2.5-0.5B-Instruct --hf_repo_id Qwen/Qwen2.5-0.5B-Instruct \
  --load_in_4bit true --max_steps 5 --skip_base_eval \
  --run_tag colab_smoke --auto_download

## 步驟 6：正式實驗

對應 TO_DO.md 第 4、5 節的 Exp 8–13。每個指令先加 `--dry_run` 看一次設定是否正確，再拿掉執行。斷線後**原指令重跑**即可續訓。
`run_tag` 決定輸出資料夾與報告檔名，換設定時一定要換 `run_tag`，不然會沿用舊的 checkpoint。

### Exp 8（L4）補 seed：1.5B，GPG 原版 vs Cooldown，各 3 個 seed

In [ ]:
for seed in [42, 123, 2026]:
    !python scripts/compare_gpg_vs_cooldown.py \
      --model_name_or_path models/Qwen2.5-1.5B-Instruct --hf_repo_id Qwen/Qwen2.5-1.5B-Instruct \
      --seed {seed} --run_tag exp8_l4_qwen15b_seed{seed} --auto_download

### Exp 9（L4）L=384 重訓並用 `max_new_tokens=512` 評測

確認 Exp6 L=384「訓練 reward 高、Eval 低」是不是評測截斷造成的。

In [ ]:
!python scripts/compare_gpg_vs_cooldown.py \
  --model_name_or_path models/Qwen2.5-1.5B-Instruct --hf_repo_id Qwen/Qwen2.5-1.5B-Instruct \
  --max_completion_length 384 --eval_max_new_tokens 512 \
  --run_tag exp9_l4_l384_eval512 --auto_download

# 已經訓練好的 run 只想換評測長度重評（不重訓），用 --eval_only 加 --report_suffix：
# !python scripts/compare_gpg_vs_cooldown.py --model_name_or_path models/Qwen2.5-1.5B-Instruct \
#   --max_completion_length 384 --run_tag exp9_l4_l384_eval512 \
#   --eval_only --eval_max_new_tokens 768 --report_suffix _eval768

### Exp 10（A100 40G）7B 診斷：加大 LoRA 與學習率

原本 7B 只從 34% 升到 37%。這組把 LoRA 掛到全部線性層、r=16，學習率提高到 1e-4，看 7B 能不能學起來。

In [ ]:
!python scripts/compare_gpg_vs_cooldown.py \
  --model_name_or_path models/Qwen2.5-7B-Instruct --hf_repo_id Qwen/Qwen2.5-7B-Instruct \
  --lora_r 16 --lora_alpha 32 --lora_target_modules all-linear --learning_rate 1e-4 \
  --run_tag exp10_a100_qwen7b_r16_lr1e4 --auto_download

### Exp 11（A100 40G）14B，4-bit QLoRA

權重約 9GB（4-bit）。4-bit 生成比 bf16 慢，若 200 步太久可先加 `--max_steps 100`。

In [ ]:
!python scripts/compare_gpg_vs_cooldown.py \
  --model_name_or_path models/Qwen2.5-14B-Instruct --hf_repo_id Qwen/Qwen2.5-14B-Instruct \
  --load_in_4bit true \
  --run_tag exp11_a100_qwen14b_4bit --auto_download

### Exp 12（A100 40G）硬體極限：G x L 掃描

每組 20 步、不評測，遇到 OOM 會記錄後跳下一組。結果在 `outputs/stress_report_<run_tag>.md`。

In [ ]:
!python scripts/stress_test_limits.py \
  --model_name_or_path models/Qwen2.5-7B-Instruct --hf_repo_id Qwen/Qwen2.5-7B-Instruct \
  --generations 4,8,16 --lengths 192,384,768 --steps 20 \
  --run_tag exp12a_a100_qwen7b --auto_download

!python scripts/stress_test_limits.py \
  --model_name_or_path models/Qwen2.5-14B-Instruct --hf_repo_id Qwen/Qwen2.5-14B-Instruct \
  --load_in_4bit true --generations 4,8,16 --lengths 192,384,768 --steps 20 \
  --run_tag exp12b_a100_qwen14b_4bit --auto_download

### Exp 13 準確率天花板：1000 步長訓練

13a 在 L4 跑（G=8, L=192），13b 在 A100 跑（再加 L=384、LoRA r=16 all-linear、評測 512 tokens）。每組約 2.5–3 小時，斷線就原指令重跑。

In [ ]:
# Exp 13a（L4）
!python scripts/compare_gpg_vs_cooldown.py \
  --model_name_or_path models/Qwen2.5-1.5B-Instruct --hf_repo_id Qwen/Qwen2.5-1.5B-Instruct \
  --max_steps 1000 --num_generations 8 --per_device_train_batch_size 8 --gradient_accumulation_steps 1 \
  --run_tag exp13a_l4_qwen15b_g8_s1000 --auto_download

# Exp 13b（A100）
# !python scripts/compare_gpg_vs_cooldown.py \
#   --model_name_or_path models/Qwen2.5-1.5B-Instruct --hf_repo_id Qwen/Qwen2.5-1.5B-Instruct \
#   --max_steps 1000 --num_generations 8 --per_device_train_batch_size 8 --gradient_accumulation_steps 1 \
#   --max_completion_length 384 --eval_max_new_tokens 512 \
#   --lora_r 16 --lora_alpha 32 --lora_target_modules all-linear \
#   --run_tag exp13b_a100_qwen15b_full_s1000 --auto_download

## 步驟 7：把報告推回 GitHub

只推 `outputs/comparison_report_*.md` 與 `.json`，checkpoint 資料夾（`outputs/exp*/`）已被 `.gitignore` 排除。
也可以在自己電腦上從 Drive 下載報告再推。

In [ ]:
!git add outputs/comparison_report_exp*.md outputs/comparison_report_exp*.json outputs/stress_report_exp*.md outputs/stress_report_exp*.json
!git -c user.name="colab" -c user.email="colab@users.noreply.github.com" commit -m "新增 Colab 實驗報告"
from google.colab import userdata
push_url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/chris456621/GPG-Cooldown.git"
!git push {push_url} HEAD